In [ ]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

In [ ]:
%pip install litellm
%pip install -U crewai

In [ ]:
import os
import requests
import litellm
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from dotenv import load_dotenv

loaded = load_dotenv()

# Configure LiteLLM
litellm.drop_params = True

# Monkey patch litellm.completion to handle parameter mapping
original_completion = litellm.completion

def patched_completion(*args, **kwargs):
    # If max_tokens is present and max_completion_tokens is not, map it
    if 'max_tokens' in kwargs and 'max_completion_tokens' not in kwargs:
        kwargs['max_completion_tokens'] = kwargs.pop('max_tokens')
    
    return original_completion(*args, **kwargs)

# Apply the patch
litellm.completion = patched_completion

In [ ]:
# Tavily API Key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)

search_tool = TavilySearchTool()

# ---- Azure LLM - FIXED ----
# Using max_tokens (not max_completion_tokens) with the monkey patch
llm = LLM(
    model=f"azure/{os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    is_litellm=True,
    temperature=1, # some models don't support temperature
    max_tokens=3500  # This will be converted to max_completion_tokens
)

In [ ]:
# Researcher agent
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=1,
    tools=[search_tool]
)

# Writer agent
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

In [ ]:
# ---- Tasks ----
task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher
)

task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 100 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer,
    context=[task_research]
)

In [ ]:
# ---- Crew ----
crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

result = await crew.kickoff_async()

print("\nFinal Output:\n")
print(result.raw)